# 03. Models

Three engines on the same folds: a lasso logistic regression as the white box, XGBoost as the machine
learning model, and TabPFN as the tabular foundation model. TabPFN needs a GPU, so it runs in
`03b_tabpfn_colab.ipynb` on Colab and comes back as a file.

All three predict **the two decisions separately and multiply them** rather than predicting the match
directly. Section 4 shows the direct approach, section 5 shows why we left it.

What matters here is not the scores but the **out-of-fold predictions**: every pair gets a prediction
from a model that never saw it. Notebooks 04 to 07 read those.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
from pathlib import Path

if not Path("data").is_dir():
    os.chdir("..")   # works whether Jupyter started at the repo root or inside notebooks/

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

table = pd.read_csv("data/processed/model_table.csv")

CATEGORICAL = ["her_field", "her_race", "her_goal", "his_field", "his_race", "his_goal"]
TARGETS = ["pair", "wave", "match", "fold", "her_dec", "his_dec"]
X = table.drop(columns=TARGETS)
y = table.match
groups = table.wave

print(X.shape[1], "columns,", len(X), "pairs,", round(y.mean(), 3), "positive")

60 columns, 4184 pairs, 0.165 positive


## 1. The folds

Notebook 02 already assigned every pair to a fold and stored it in the table, so every model here is
scored on the same split, and so is TabPFN, which reads the same file in Colab.

In [2]:
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)   # the one 02 used, kept for reference

table.groupby("fold").agg(pairs=("match", "size"), matches=("match", "sum"),
                          rate=("match", "mean"), waves=("wave", "nunique")).round(3)

,pairs,matches,rate,waves
fold,,,,
0,847,137,0.162,4
1,877,140,0.160,4
2,827,122,0.148,4
3,823,133,0.162,5
4,810,158,0.195,4


## 2. The white box: logistic regression

The brief asks for a white-box model and the obvious one is a logistic regression: one coefficient
per feature, readable signs, and a prediction any analyst can reproduce by hand.

The only real choice is the penalty. All of them score the same here, so we pick on sparsity rather
than on decimals.

In [3]:
def encoded(final_step):
    """One-hot the six category codes on the training fold only, then impute, scale, and fit."""
    encode = ColumnTransformer(
        [("categories", OneHotEncoder(min_frequency=0.03, handle_unknown="infrequent_if_exist",
                                      sparse_output=False), CATEGORICAL)],
        remainder="passthrough")
    return make_pipeline(encode, SimpleImputer(strategy="median"), StandardScaler(), final_step)

penalties = {
    "ridge (L2), C=1":   LogisticRegression(max_iter=3000, C=1, random_state=0),
    "lasso (L1), C=0.1":  LogisticRegression(max_iter=3000, C=0.1, l1_ratio=1, solver="liblinear", random_state=0),
    "lasso (L1), C=0.05": LogisticRegression(max_iter=3000, C=0.05, l1_ratio=1, solver="liblinear", random_state=0),
}

rows = []
for name, est in penalties.items():
    pipe = encoded(est)
    auc = cross_val_score(pipe, X, y, cv=cv, groups=groups, scoring="roc_auc")
    pipe.fit(X, y)
    rows.append({"penalty": name, "ROC-AUC": round(auc.mean(), 3),
                 "features kept": int((pipe[-1].coef_ != 0).sum())})

pd.DataFrame(rows).set_index("penalty")

,ROC-AUC,features kept
penalty,,
"ridge (L2), C=1",0.581,99
"lasso (L1), C=0.1",0.579,59
"lasso (L1), C=0.05",0.579,45


The lasso at `C=0.05` keeps half the features for three thousandths of AUC, well inside the
fold-to-fold noise. Half as many coefficients in front of the client is worth more than that, so this
is our white box. 04 reads them one by one, on the two-stage model we actually ship.

## 3. XGBoost

Same pipeline, without the scaling it does not need. The settings are deliberately modest: 4,184 rows
with a 16.5% positive rate is not much, and a deeper model only memorises the training waves.

In [4]:
def boosted():
    encode = ColumnTransformer(
        [("categories", OneHotEncoder(min_frequency=0.03, handle_unknown="infrequent_if_exist",
                                      sparse_output=False), CATEGORICAL)],
        remainder="passthrough")
    return make_pipeline(encode, SimpleImputer(strategy="median"),
                         XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                       subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                                       random_state=0))

## 4. The obvious approach, and its baseline

The straightforward thing is to fit on the pairs and predict `match`. One pass over the folds: each
model is fitted on four and predicts the fifth, so every pair gets a prediction from a model that
never saw it. This is our reference point.

In [5]:
DIRECT = {"logit_direct": lambda: encoded(LogisticRegression(max_iter=3000, C=0.05, l1_ratio=1,
                                                             solver="liblinear", random_state=0)),
          "xgboost_direct": boosted}

oof = table[["pair", "wave", "fold", "match"]].copy()

for k in sorted(table.fold.unique()):
    train, test = table.fold != k, table.fold == k
    for name, build in DIRECT.items():
        model = build().fit(X[train], y[train])
        oof.loc[test, name] = model.predict_proba(X[test])[:, 1]

print("predictions missing:", oof[list(DIRECT)].isna().sum().sum())
oof.head(3)

predictions missing: 0


,pair,wave,fold,match,logit_direct,xgboost_direct
0,"(1, 11)",1,1,0,0.106454,0.045065
1,"(1, 12)",1,1,0,0.362237,0.211066
2,"(1, 13)",1,1,1,0.172201,0.103035


## 5. What we actually use: a match is two decisions

Predicting `match` directly asks the model to hit a rare joint event, 16.5% of pairs. But a match is
she says yes **and** he says yes, and each decision is far more common on its own: 37% for her, 47%
for him. So one model on 8,368 decisions instead of one on 4,184 pairs, and the two probabilities are
multiplied.

Twice the rows, a nearly balanced target, and the asymmetry between the two sides carried by a
`female` column. `dec` is unusable as a feature, since it is the answer, but it is a legitimate
target.

In [6]:
# One row per decision: whoever is deciding becomes `self_`, their partner `other_`.
def decision_view(tab, decider):
    mine, theirs = ("her", "his") if decider == "her" else ("his", "her")
    view = tab.drop(columns=TARGETS).rename(
        columns=lambda c: c.replace(f"{mine}_", "self_").replace(f"{theirs}_", "other_"))
    view["agediff"] = view.self_age - view.other_age   # directional: follows whoever is deciding
    view["female"] = int(decider == "her")
    return view

SIDE_CATEGORICAL = ["self_field", "self_race", "self_goal", "other_field", "other_race", "other_goal"]

hers, his = decision_view(table, "her"), decision_view(table, "his")
print(hers.shape, "rows per side ->", 2 * len(hers), "decisions in total")

(4184, 61) rows per side -> 8368 decisions in total


In [7]:
def two_stage(final_step, scale):
    encode = ColumnTransformer(
        [("categories", OneHotEncoder(min_frequency=0.03, handle_unknown="infrequent_if_exist",
                                      sparse_output=False), SIDE_CATEGORICAL)],
        remainder="passthrough")
    steps = [encode, SimpleImputer(strategy="median")]
    if scale:
        steps.append(StandardScaler())
    return make_pipeline(*steps, final_step)

MODELS = {
    "logit": lambda: two_stage(LogisticRegression(max_iter=3000, C=0.05, l1_ratio=1,
                                                  solver="liblinear", random_state=0), scale=True),
    "xgboost": lambda: two_stage(XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                               subsample=0.8, colsample_bytree=0.8,
                                               eval_metric="logloss", random_state=0), scale=False),
}

for name, build in MODELS.items():
    oof[name] = np.nan
    for k in sorted(table.fold.unique()):
        train, test = table.fold != k, table.fold == k
        model = build().fit(pd.concat([hers[train], his[train]]),
                            pd.concat([table.her_dec[train], table.his_dec[train]]))
        oof.loc[test, name] = (model.predict_proba(hers[test])[:, 1]
                               * model.predict_proba(his[test])[:, 1])

print("predictions missing:", oof[list(MODELS)].isna().sum().sum())

predictions missing: 0


## 6. Where we stand

ROC-AUC says how well the pairs are ranked overall. PR-AUC is the one to watch here, because only
16.5% of pairs match and a model can look respectable on ROC-AUC while being useless at the top of
the ranking, which is the only part the app would ever show.

In [8]:
def score(column):
    per_fold = [roc_auc_score(g.match, g[column]) for _, g in oof.groupby("fold")]
    return {"ROC-AUC": round(roc_auc_score(oof.match, oof[column]), 3),
            "PR-AUC": round(average_precision_score(oof.match, oof[column]), 3),
            "worst fold": round(min(per_fold), 3),
            "best fold": round(max(per_fold), 3)}

ALL = list(MODELS) + list(DIRECT)   # the two-stage models first, the baseline after

results = pd.DataFrame({name: score(name) for name in ALL}).T
results.index.name = "model"
results

,ROC-AUC,PR-AUC,worst fold,best fold
model,,,,
logit,0.587,0.228,0.552,0.618
xgboost,0.585,0.218,0.553,0.612
logit_direct,0.573,0.203,0.532,0.635
xgboost_direct,0.589,0.212,0.568,0.635


In [9]:
# What the client would actually see: if the app shows the top 10% of pairs, how many are matches?
base = oof.match.mean()
rows = []
for name in ALL:
    top = oof.nlargest(int(0.10 * len(oof)), name)
    rows.append({"model": name, "match rate in the top 10%": round(top.match.mean(), 3),
                 "lift over showing pairs at random": round(top.match.mean() / base, 2)})

pd.DataFrame(rows).set_index("model")

,match rate in the top 10%,lift over showing pairs at random
model,,
logit,0.299,1.81
xgboost,0.285,1.73
logit_direct,0.237,1.44
xgboost_direct,0.230,1.39


**This is the table that decides the approach.** Splitting the match into two decisions costs nothing
on ROC-AUC and gains a third on the metric that matters: 29.8% of the top decile match against 23.6%
predicting the match directly, on a base rate of 16.5%.

It also says something the slides should not bury: **ROC-AUC ranks the models in almost the reverse
order of the metric the client cares about.** Direct XGBoost has the best ROC-AUC and the worst top
decile.

### How much of that is real?

Every number above comes from 4,184 pairs, and the top decile is measured on 418 of them. Before
reading anything into a gap of two points, we need to know how wide the error bars are.

In [10]:
rng = np.random.default_rng(0)

rows = []
for name in ALL:
    draws = []
    for _ in range(2000):
        sample = oof.iloc[rng.integers(0, len(oof), len(oof))]
        draws.append(sample.match[sample[name] >= sample[name].quantile(0.9)].mean())
    low, high = np.percentile(draws, [2.5, 97.5])
    rows.append({"model": name, "top 10%": round(np.mean(draws), 3),
                 "95% interval": f"[{low:.3f}, {high:.3f}]", "width": round(high - low, 3)})

pd.DataFrame(rows).set_index("model")

,top 10%,95% interval,width
model,,,
logit,0.296,"[0.250, 0.340]",0.090
xgboost,0.284,"[0.241, 0.327]",0.086
logit_direct,0.232,"[0.193, 0.274]",0.081
xgboost_direct,0.229,"[0.191, 0.269]",0.078


**What it says.** The interval is about eight points wide, and two things follow.

The two-stage gain is **real**: roughly 0.30 against 0.23, on intervals that barely meet. The gap
between the two-stage models is **not**: 0.30 against 0.29 with intervals that overlap almost
entirely.

**Conclusion.** Performance cannot separate these models. We chased it further, with eighteen more
engineered features, and every variant landed inside the same interval, so we stopped. Interpretability,
stability and fairness will have to decide, which is what 04 to 07 do.

In [11]:
oof.to_csv("data/processed/oof_predictions.csv", index=False)
print("saved", oof.shape, "to data/processed/oof_predictions.csv")

saved (4184, 8) to data/processed/oof_predictions.csv


## 7. TabPFN

TabPFN runs in `03b_tabpfn_colab.ipynb` on a Colab GPU, on the same five folds with the same two-stage
target, so the three engines are directly comparable. It writes `data/processed/tabpfn_oof.csv`, and
the cell below merges it in.

In [12]:
from pathlib import Path

tabpfn_file = Path("data/processed/tabpfn_oof.csv")
if tabpfn_file.exists():
    arrived = pd.read_csv(tabpfn_file).set_index("pair")
    for column in ["tabpfn", "tabpfn_direct"]:
        oof[column] = arrived[column].reindex(oof.pair).to_numpy()
    oof.to_csv("data/processed/oof_predictions.csv", index=False)
    print(pd.DataFrame({n: score(n) for n in ["logit", "xgboost", "tabpfn",
                                              "logit_direct", "xgboost_direct", "tabpfn_direct"]}).T)
else:
    print("tabpfn_oof.csv not here yet: run 03b in Colab, then re-run this cell.")

                ROC-AUC  PR-AUC  worst fold  best fold
logit             0.587   0.228       0.552      0.618
xgboost           0.585   0.218       0.553      0.612
tabpfn            0.600   0.234       0.578      0.633
logit_direct      0.573   0.203       0.532      0.635
xgboost_direct    0.589   0.212       0.568      0.635
tabpfn_direct     0.597   0.223       0.555      0.633


## What the next notebooks get

`data/processed/oof_predictions.csv`: one row per pair with `wave`, `fold`, `match` and one
out-of-fold probability per engine. `logit`, `xgboost` and `tabpfn` are the two-stage models;
the `*_direct` columns are the baseline, kept so the choice can be defended.

- **04** interpretability, **05** stability, **06** fairness, **07** the scorecard and the
  recommendation.
- Note for 04: the two-stage models explain a **decision**, not a match, which is easier to present
  to the client. "Why would she say yes to him".
- Note for 06: ethnicity is in the table on both sides, and `raceclash` is built from it.